# ARC-AGI-2 program synthesis on Colab

Runtime: A100 (80GB preferred). Upload `arc-agi_evaluation_challenges.json` and `arc-agi_evaluation_solutions.json` to `/content/data/` first (or mount Drive and copy them).

In [ ]:
MODEL = 'Qwen/Qwen3.8-27B-FP8'   # 80GB A100. On a 40GB A100 use 'cyankiwi/Qwen3.8-27B-AWQ-INT4'
SUBSET = 'screen8'               # 'screen8', 'timing20' or 'dev'
SAMPLES, REPAIRS, MAX_TOKENS, PARALLEL = 4, 1, 12000, 8

In [ ]:
!pip -q install vllm
!mkdir -p /content/data /content/work/induction /content/runs

In [ ]:
import json, os
split = {"screen8": ["4c7dc4dd", "71e489b6", "7b3084d4", "8b9c3697", "b6f77b65", "c4d067a0", "dbff022c", "eee78d87"], "timing20": ["195c6913", "21897d95", "2c181942", "409aa875", "4c7dc4dd", "58490d8a", "5961cc34", "71e489b6", "7491f3cf", "7b3084d4", "8b9c3697", "8e5c0c38", "a6f40cea", "b6f77b65", "c4d067a0", "da515329", "dbff022c", "eee78d87", "f931b4a8", "faa9f03d"], "dev": ["0934a4d8", "13e47133", "142ca369", "16b78196", "1818057f", "195c6913", "21897d95", "221dfab4", "269e22fb", "28a6681f", "2b83f449", "2c181942", "2d0172a1", "31f7f899", "332f06d7", "35ab12c3", "3a25b0d8", "3dc255db", "3e6067c3", "409aa875", "446ef5d2", "45a5af55", "4c3d4a41", "4c7dc4dd", "4e34c42c", "53fb4810", "581f7754", "58490d8a", "5961cc34", "62593bfd", "67e490f4", "6e453dd6", "6ffbe589", "71e489b6", "7491f3cf", "7666fa5d", "78332cb0", "7b3084d4", "7b5033c1", "7b80bb43", "7ed72f31", "80a900e0", "88e364bc", "89565ca0", "8b7bacbf", "8b9c3697", "8e5c0c38", "8f215267", "8f3a5a89", "9385bd28", "97d7923e", "9bbf930d", "a251c730", "a47bf94d", "a6f40cea", "b5ca7ac4", "b6f77b65", "b9e38dc0", "c4d067a0", "c7f57c3e", "cbebaa4b", "d35bdbdc", "d59b0160", "d8e07eb2", "da515329", "db0c5428", "db695cfb", "dbff022c", "de809cff", "dfadab01", "e12f9a14", "e376de54", "e8686506", "e87109e9", "edb79dae", "eee78d87", "f560132c", "f931b4a8", "faa9f03d", "fc7cae8d"]}
os.makedirs('/content/splits', exist_ok=True)
json.dump(split, open('/content/splits/eval_split.json', 'w'))
assert os.path.exists('/content/data/arc-agi_evaluation_challenges.json'), 'upload the evaluation json files to /content/data'

In [ ]:
%%writefile /content/work/induction/sandbox.py
"""Run a generated `transform(grid)` on grids in a separate process with time and memory limits."""
import json
import subprocess
import sys

RUNNER = r'''
import json, sys
try:
    import resource
    limit = int(sys.argv[1]) * 1024 * 1024
    resource.setrlimit(resource.RLIMIT_AS, (limit, limit))
except Exception:
    pass  # not available on Windows
payload = json.load(sys.stdin)
results = []
try:
    scope = {}
    exec(payload["code"], scope)
    transform = scope["transform"]
except BaseException as e:
    print(json.dumps({"error": f"{type(e).__name__}: {e}"}))
    sys.exit(0)
import copy
for grid in payload["grids"]:
    try:
        out = transform(copy.deepcopy(grid))
        if hasattr(out, "tolist"):
            out = out.tolist()
        out = [[int(c) for c in row] for row in out]
        ok = 0 < len(out) <= 30 and 0 < len(out[0]) <= 30 and all(len(r) == len(out[0]) for r in out) and all(0 <= c <= 9 for r in out for c in r)
        results.append(out if ok else {"error": "output is not a rectangular grid of digits 0-9 up to 30x30"})
    except BaseException as e:
        results.append({"error": f"{type(e).__name__}: {e}"})
print(json.dumps({"results": results}))
'''


def run_program(code, grids, timeout=4.0, memory_mb=2048):
    """Returns one entry per grid: the output grid, or {"error": ...}."""
    try:
        proc = subprocess.run(
            [sys.executable, "-I", "-c", RUNNER, str(memory_mb)],
            input=json.dumps({"code": code, "grids": grids}),
            capture_output=True, text=True, timeout=timeout,
        )
        out = json.loads(proc.stdout.strip().splitlines()[-1])
    except subprocess.TimeoutExpired:
        return [{"error": "timeout"}] * len(grids)
    except Exception as e:
        return [{"error": f"runner failed: {type(e).__name__}"}] * len(grids)
    if "error" in out:
        return [{"error": out["error"]}] * len(grids)
    return out["results"]


In [ ]:
%%writefile /content/work/induction/solver.py
"""Program synthesis leg: ask a code model for `transform(grid)`, keep programs that reproduce every train pair."""
import json
import re
import time
import urllib.request

from sandbox import run_program

SYSTEM = (
    "You solve ARC puzzles by writing Python. Each puzzle has input/output grid pairs that all follow one rule. "
    "Grids are lists of lists of integers 0-9 (colours). Work out the rule, then write a function "
    "`transform(grid)` that takes an input grid (list of lists of int) and returns the output grid. "
    "The function must be general: it is checked on the examples and then applied to new inputs. "
    "Do not hard-code the example outputs. You may use numpy and the standard library. "
    "Reason briefly first, then give the complete code in one ```python block."
)


def grid_text(grid):
    return "\n".join("".join(str(c) for c in row) for row in grid)


def task_prompt(task):
    parts = []
    for i, ex in enumerate(task["train"], 1):
        parts.append(f"Example {i} input ({len(ex['input'])}x{len(ex['input'][0])}):\n{grid_text(ex['input'])}\n"
                     f"Example {i} output ({len(ex['output'])}x{len(ex['output'][0])}):\n{grid_text(ex['output'])}")
    for i, ex in enumerate(task["test"], 1):
        parts.append(f"Test input {i} ({len(ex['input'])}x{len(ex['input'][0])}):\n{grid_text(ex['input'])}")
    return "\n\n".join(parts) + "\n\nWrite `transform(grid)`."


def feedback_prompt(task, outputs):
    lines = ["Your function does not reproduce the examples:"]
    for i, (ex, out) in enumerate(zip(task["train"], outputs), 1):
        if out == ex["output"]:
            lines.append(f"Example {i}: correct.")
        elif isinstance(out, dict):
            lines.append(f"Example {i}: {out['error']}")
        else:
            wrong = sum(a != b for ra, rb in zip(out, ex["output"]) for a, b in zip(ra, rb)) if (len(out), len(out[0])) == (len(ex["output"]), len(ex["output"][0])) else None
            detail = f"{wrong} cells differ" if wrong is not None else f"shape {len(out)}x{len(out[0])} instead of {len(ex['output'])}x{len(ex['output'][0])}"
            lines.append(f"Example {i}: wrong ({detail}). Your output:\n{grid_text(out)}")
    lines.append("Reconsider the rule and give the corrected complete code in one ```python block.")
    return "\n".join(lines)


def extract_code(text):
    blocks = re.findall(r"```(?:python)?\n(.*?)```", text, flags=re.S)
    for block in reversed(blocks):
        if "def transform" in block:
            return block
    return None


CONTEXT_LIMIT = int(__import__("os").getenv("ARC_LLM_CONTEXT", "0"))  # 0 = let the server decide


def chat(base_url, model, messages, n, temperature, max_tokens, timeout=3600):
    if CONTEXT_LIMIT:
        # Rough token estimate so that prompt + answer fits a small context window.
        prompt_tokens = sum(len(m["content"]) for m in messages) // 3
        max_tokens = max(256, min(max_tokens, CONTEXT_LIMIT - prompt_tokens - 64))
    body = json.dumps({"model": model, "messages": messages, "n": n, "temperature": temperature, "max_tokens": max_tokens}).encode()
    req = urllib.request.Request(base_url + "/chat/completions", data=body, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        out = json.load(resp)
    return [c["message"]["content"] or "" for c in out["choices"]], out.get("usage", {})


def solve_task(task, base_url, model, n_samples=4, n_repairs=1, temperature=0.8, max_tokens=1500, prog_timeout=4.0):
    """Returns {"programs": [...], "verified": [test outputs per verified program], "seconds", "tokens"}."""
    start = time.time()
    train_inputs = [ex["input"] for ex in task["train"]]
    test_inputs = [ex["input"] for ex in task["test"]]
    base = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": task_prompt(task)}]
    record = {"programs": [], "verified": [], "tokens": 0}

    def check(code, stage):
        outputs = run_program(code, train_inputs + test_inputs, timeout=prog_timeout)
        train_out, test_out = outputs[:len(train_inputs)], outputs[len(train_inputs):]
        n_ok = sum(o == ex["output"] for o, ex in zip(train_out, task["train"]))
        ok = n_ok == len(train_inputs) and not any(isinstance(o, dict) for o in test_out)
        record["programs"].append({"stage": stage, "code": code, "train_ok": n_ok, "verified": ok})
        if ok:
            record["verified"].append(test_out)
        return ok, train_out

    replies, usage = chat(base_url, model, base, n_samples, temperature, max_tokens)
    record["tokens"] += usage.get("total_tokens", 0)
    failures = []
    for reply in replies:
        code = extract_code(reply)
        if code is None:
            record["programs"].append({"stage": "sample", "code": None, "train_ok": 0, "verified": False})
            continue
        ok, train_out = check(code, "sample")
        if not ok:
            n_ok = record["programs"][-1]["train_ok"]
            failures.append((n_ok, reply, train_out))

    # Repair only when nothing verified, starting from the attempts that got the most examples right.
    if not record["verified"]:
        for _, reply, train_out in sorted(failures, key=lambda f: -f[0])[:2] * n_repairs:
            messages = base + [{"role": "assistant", "content": reply}, {"role": "user", "content": feedback_prompt(task, train_out)}]
            fixes, usage = chat(base_url, model, messages, 1, 0.4, max_tokens)
            record["tokens"] += usage.get("total_tokens", 0)
            code = extract_code(fixes[0])
            if code is not None and check(code, "repair")[0]:
                break

    record["seconds"] = time.time() - start
    return record


In [ ]:
%%writefile /content/work/run_induction.py
"""Run the program-synthesis leg against a local OpenAI-compatible server (vLLM) and score it.

    uv run --no-project python scripts/run_induction.py <run-name> --subset screen8 --model <served-model-name>
"""
import argparse
import json
import pathlib
import sys
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

ROOT = pathlib.Path('/content')
sys.path.insert(0, str(ROOT / "work" / "induction"))
from solver import solve_task  # noqa: E402

parser = argparse.ArgumentParser()
parser.add_argument("name")
parser.add_argument("--subset", default="screen8")
parser.add_argument("--model", required=True)
parser.add_argument("--base-url", default="http://localhost:8000/v1")
parser.add_argument("--samples", type=int, default=4)
parser.add_argument("--repairs", type=int, default=1)
parser.add_argument("--max-tokens", type=int, default=1500)
parser.add_argument("--parallel", type=int, default=4)
args = parser.parse_args()

split = json.load(open(ROOT / "splits" / "eval_split.json"))
keys = split[args.subset] if args.subset in split else args.subset.split(",")
tasks = json.load(open(ROOT / "data" / "arc-agi_evaluation_challenges.json"))
solutions = json.load(open(ROOT / "data" / "arc-agi_evaluation_solutions.json"))
out_dir = pathlib.Path("/content/runs") / args.name
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "config.json").write_text(json.dumps(vars(args) | {"keys": keys}, indent=1))


def run(key):
    path = out_dir / f"{key}.json"
    if path.exists():
        return key, json.loads(path.read_text())
    try:
        record = solve_task(tasks[key], args.base_url, args.model, args.samples, args.repairs, max_tokens=args.max_tokens)
    except Exception as e:
        record = {"programs": [], "verified": [], "tokens": 0, "seconds": 0, "error": f"{type(e).__name__}: {e}"}
    path.write_text(json.dumps(record))
    return key, record


with ThreadPoolExecutor(args.parallel) as pool:
    results = dict(pool.map(run, keys))

print(f"{'task':10} {'sec':>5} {'progs':>5} {'best':>5} {'verif':>5} {'correct':>7}  note")
total = verified_tasks = false_verified = 0.0
for key in keys:
    r = results[key]
    n_test = len(solutions[key])
    # Verified programs vote on each test output; the two most common outputs are the attempts.
    score = 0.0
    for i in range(n_test):
        votes = Counter(json.dumps(v[i]) for v in r["verified"])
        attempts = [json.loads(g) for g, _ in votes.most_common(2)]
        score += any(a == solutions[key][i] for a in attempts) / n_test
    total += score
    verified_tasks += bool(r["verified"])
    false_verified += bool(r["verified"]) and score == 0
    best = max([p["train_ok"] for p in r["programs"]], default=0)
    print(f"{key:10} {r['seconds']:5.0f} {len(r['programs']):5d} {best:>3}/{len(tasks[key]['train'])} {len(r['verified']):5d} {score:7.2f}  {r.get('error', '')}")

n = len(keys)
print(f"\ntasks {n}  solved {total:.2f} ({100 * total / n:.1f}%)  with a verified program {verified_tasks:.0f}  verified but wrong {false_verified:.0f}")
print(f"mean seconds {sum(r['seconds'] for r in results.values()) / n:.0f}  total tokens {sum(r['tokens'] for r in results.values())}")


In [ ]:
import subprocess, time, urllib.request
server = subprocess.Popen(['python', '-m', 'vllm.entrypoints.openai.api_server', '--model', MODEL, '--served-model-name', 'llm', '--max-model-len', '32768', '--gpu-memory-utilization', '0.92', '--max-num-seqs', str(PARALLEL)], stdout=open('/content/vllm.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(240):
    time.sleep(10)
    try:
        urllib.request.urlopen('http://localhost:8000/v1/models', timeout=3); print('server ready'); break
    except Exception:
        if server.poll() is not None: print(open('/content/vllm.log').read()[-3000:]); raise SystemExit('vllm died')

In [ ]:
!cd /content && python work/run_induction.py ind_{MODEL.split('/')[-1]}_{SUBSET} --subset {SUBSET} --model llm --samples {SAMPLES} --repairs {REPAIRS} --max-tokens {MAX_TOKENS} --parallel {PARALLEL}

In [ ]:
# Download the results folder (or copy it to Drive) and put it under ~/arc2-local/runs/ locally.
!cd /content/runs && zip -qr /content/induction_results.zip . && ls -la /content/induction_results.zip